# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and preview its main description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset title and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
List available record sets and their `@id`, overview their fields and columns using the Croissant metadata.


In [ ]:
# Helper function to extract entities by type
def find_entities_by_type(metadata_json, entity_type):
    entities = []
    def search(obj):
        if isinstance(obj, dict):
            if '@type' in obj and (obj['@type']==entity_type or (isinstance(obj['@type'], list) and entity_type in obj['@type'])):
                entities.append(obj)
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for v in obj:
                search(v)
    search(dataset.metadata.to_json())  # traverses deep JSON
    return entities

# Find RecordSets
record_sets = find_entities_by_type(dataset.metadata.to_json(), 'cr:RecordSet')
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        rs_id = rs.get('@id', None)
        print(f"RecordSet @id: {rs_id}")
        print(f"  Name: {rs.get('name', '<no name>')}")
        print(f"  Description: {rs.get('description', '<no description>')}")
        # List fields/columns
        columns = rs.get('cr:column', []) or rs.get('column', [])
        if columns:
            print(f"  Columns:")
            for field in columns:
                if isinstance(field, dict):
                    field_id = field.get('@id', None)
                else:
                    field_id = field
                print(f"    - {field_id}")
        print("")

**Print a preview record from each record set (by `@id`) using mlcroissant.**

In [ ]:
# We'll demonstrate using the first found RecordSet (if available)
if not record_sets:
    print("No record sets to preview records from.")
else:
    for rs in record_sets:
        rs_id = rs.get('@id')
        print(f"\nPreview of records from RecordSet @id: {rs_id}")
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i>=2:
                print("(Showed 3 records)")
                break

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames. Reference all entities by their `@id` fields for clarity.

In [ ]:
# Collect all RecordSet @ids
record_set_ids = [rs.get('@id') for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {rs_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for RecordSet @id {rs_id}")
        print(e)

if dataframes:
    # Just pick the first available for further exploration
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nPrimary table for exploration: RecordSet @id = {main_rs_id}")
    print(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate common data analysis tasks. Let's select a numeric field by its `@id` for filtering, normalization, and grouping.

In [ ]:
# Attempt to pick a numeric field for analysis, e.g. `Coverage(%)`, `MW`, `pI`, `Unique Peptides`, `Total Peptides`, `Abundance`

numeric_candidates = ['Coverage(%)', 'MW', 'pI', 'Unique Peptides', 'Total Peptides', 'Abundance', 'Abundance-normalized']
df = dataframes[main_rs_id]
numeric_field = None
for c in numeric_candidates:
    if c in df.columns:
        numeric_field = c
        break

if numeric_field is not None:
    print(f"Selected numeric field: {numeric_field}")
    # Remove obvious non-numeric values
    df_valid = df.copy()
    df_valid[numeric_field] = pd.to_numeric(df_valid[numeric_field], errors='coerce')
    threshold = df_valid[numeric_field].mean()  # Example: use mean as cutoff
    filtered_df = df_valid[df_valid[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df[[numeric_field]].head())

    # Normalize field (z-scoring)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    group_field_candidates = ['Sample', 'Accession', 'Description']
    group_field = None
    for c in group_field_candidates:
        if c in df.columns:
            group_field = c
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped mean {numeric_field} by '{group_field}':")
        print(grouped_df.head())
else:
    print("No obvious numeric field found for analysis.")

## 5. Visualization
Plot a histogram of the main numeric field and a boxplot grouped by a categorical field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(df_valid[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=df_valid)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
- We loaded the [FAIR^2 EV Mass Spec](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using `mlcroissant` and explored its structure by referencing all record sets and fields by their `@id`.
- We demonstrated extraction to Pandas DataFrames, performed basic filtering and normalization, and visualized one numeric attribute.
- All exploration referenced dataset entities by their Croissant `@id` for clarity and reproducibility in future workflows.